# 011-uncertain-band-refinement

# Imports

In [ ]:
import json
from dataclasses import asdict, replace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from uavloc.data.annotated import IGNORE_LABEL
from uavloc.datasets import dataset_ref
from uavloc.features.segmentation import CLASSES
from uavloc.plotting.correlation import SEG_PALETTE
from uavloc.segmentation.boundary import label_band
from uavloc.segmentation.eval import (MEAN_KEY, VAL_SETS, MaskedRefineFn, RefineFn, bilinear_argmax,
                                      evaluate, evaluate_predictions, load_baseline_model, load_val_sets,
                                      lraspp_outputs, results_table, run_model)
from uavloc.segmentation.latency import refiner_latency, standard_tile
from uavloc.segmentation.refine import dense_crf, guided_filter
from uavloc.segmentation.refine.band import BandConfig, band_fractions, band_refiner, uncertain_band
from uavloc.segmentation.refine.superpixels import (SuperpixelMethod, VoteConfig, masked_superpixel_refiner,
                                                    superpixel_refiner)
from uavloc.track import experiment_dir, log_run, set_seed

# Params

In [ ]:
EXP = "011-uncertain-band-refinement"
NOTE = ""
LOG_RUN = True

REFINER = "baseline"   # "baseline" | "guided" | "crf_best" | "crf_compat10" | "watershed"
GUIDED = {"mode": "fast", "guide": "rgb", "radius": 1, "eps": 1e-3}
CRF_BEST = {"iterations": 5, "bilateral_sxy_px": 20.0, "bilateral_srgb": 5.0, "bilateral_compat": 3.0,
            "gaussian_sxy_px": 3.0, "gaussian_compat": 3.0}
CRF_COMPAT10 = {**CRF_BEST, "bilateral_compat": 10.0}
WATERSHED = {"cell_px": 6, "vote": "all", "alpha": 1.0}

BAND_DISTANCES_PX = [3, 6, 12]
BAND_MARGINS = [0.2, 0.5]
TILE_PX = 96
CONTEXT_PX = 24

TUNE_SET = "birky-strip"
MIOU_GUARD = 0.01
BIOU_BAR = 0.010
LATENCY_AREA_SLACK = 0.15

DATASET = "ua-village-annotated"
SEED = 42
DEVICE = "cpu"
LATENCY_SET = "bushcha"
LATENCY_TOP_LEFT_PX = [1024, 1024]
LATENCY_THREADS = 4
LATENCY_WARMUP = 2
LATENCY_RUNS = 10
ZOOMS = [["birky-strip-google", 0, [120, 1400, 376, 1656]], ["birky-strip", 3, [200, 2300, 456, 2556]],
         ["bushcha", 0, [1100, 1100, 1356, 1356]]]   # set, image index, [top, left, bottom, right] px

# Data

In [ ]:
set_seed(SEED)
ARTIFACTS = experiment_dir(EXP) / "artifacts"

In [ ]:
val_sets = load_val_sets(DATASET)
{name: [a.rgb.shape[:2] for a in items] for name, items in val_sets.items()}

# Model

In [ ]:
model = load_baseline_model(DEVICE)

In [ ]:
cases = run_model(model, val_sets, DEVICE)

In [ ]:
def base_refiners(name: str) -> tuple[RefineFn, MaskedRefineFn] | None:
    """The full-frame and the per-pixel version of one registered base refiner; None for the baseline."""
    if name == "baseline":
        return None
    if name == "guided":
        config = guided_filter.GuidedFilterConfig(**GUIDED)
        return guided_filter.refiner(config), guided_filter.masked_refiner(config)
    if name in ("crf_best", "crf_compat10"):
        config = dense_crf.DenseCrfConfig(**(CRF_BEST if name == "crf_best" else CRF_COMPAT10))
        return dense_crf.dense_crf_refiner(config), dense_crf.masked_dense_crf_refiner(config)
    if name == "watershed":
        method, vote = SuperpixelMethod("watershed", WATERSHED["cell_px"]), VoteConfig(WATERSHED["vote"], WATERSHED["alpha"])
        return superpixel_refiner(method, vote), masked_superpixel_refiner(method, vote)
    raise ValueError(f"unknown refiner {name!r}")

In [ ]:
refiners = base_refiners(REFINER)
full_fn, masked_fn = refiners if refiners else (bilinear_argmax, None)

In [ ]:
def band_configs() -> list[BandConfig]:
    """The registered band grid in pixel mode; empty for the baseline."""
    if REFINER == "baseline":
        return []
    return ([BandConfig(float(d), None, "pixels", TILE_PX, CONTEXT_PX) for d in BAND_DISTANCES_PX]
            + [BandConfig(None, float(m), "pixels", TILE_PX, CONTEXT_PX) for m in BAND_MARGINS])

In [ ]:
def band_name(config: BandConfig) -> str:
    return f"d{config.distance_px:g}" if config.distance_px is not None else f"m{config.margin:g}"

# Run

In [ ]:
baseline = evaluate(bilinear_argmax, cases)
results_table(baseline).round(4)

In [ ]:
full_preds = {name: [full_fn(c.output, c.image.rgb) for c in items] for name, items in cases.items()}
full = evaluate_predictions(full_preds, cases)
results_table(full).round(4)

In [ ]:
def scalar_row(scores: dict, prefix: str = "") -> dict:
    """Scalar metrics plus per-class IoU and Boundary IoU as flat columns."""
    row = {f"{prefix}{k}": v for k, v in scores.items() if not isinstance(v, dict)}
    for metric in ("iou", "biou"):
        row.update({f"{prefix}{metric}_{c}": v for c, v in scores.get(f"{metric}_per_class", {}).items()})
    return row

In [ ]:
def band_predictions(config: BandConfig, set_names: list[str]) -> tuple[dict, dict, dict]:
    """Per set: pixel-mode band predictions, frame-masked band predictions, band area fractions."""
    pixels, frame, area = {}, {}, {}
    refine_fn = band_refiner(masked_fn, config)
    for name in set_names:
        items = cases[name]
        pixels[name] = [refine_fn(c.output, c.image.rgb) for c in items]
        bands = [uncertain_band(c.output, config) for c in items]
        frame[name] = [np.where(b, f, bilinear_argmax(c.output, c.image.rgb)).astype(np.uint8)
                       for b, f, c in zip(bands, full_preds[name], items)]
        area[name] = float(np.mean([b.mean() for b in bands]))
    return pixels, frame, area

In [ ]:
def grid_row(config: BandConfig) -> dict:
    """One band config scored on TUNE_SET, in pixel mode and as the masked full-frame output."""
    pixels, frame, area = band_predictions(config, [TUNE_SET])
    tune = {TUNE_SET: cases[TUNE_SET]}
    return {"band": band_name(config), **asdict(config), "area": area[TUNE_SET],
            **scalar_row(evaluate_predictions(pixels, tune)[TUNE_SET]),
            **scalar_row(evaluate_predictions(frame, tune)[TUNE_SET], "frame_")}

In [ ]:
configs = band_configs()
grid = pd.DataFrame([grid_row(c) for c in configs])
tune_baseline, tune_full = scalar_row(baseline[TUNE_SET]), scalar_row(full[TUNE_SET])
grid[["band", "area", "miou", "biou", "bf1_1px", "frame_miou", "frame_biou", "iou_track"]].round(4) if len(grid) else grid

In [ ]:
def select(grid: pd.DataFrame) -> BandConfig | None:
    """Highest TUNE_SET Boundary IoU among band configs within MIOU_GUARD of the baseline's TUNE_SET mIoU."""
    eligible = grid[grid["miou"] >= tune_baseline["miou"] - MIOU_GUARD] if len(grid) else grid
    if eligible.empty:
        return None
    return configs[int(eligible["biou"].idxmax())]

In [ ]:
chosen = select(grid)
chosen

In [ ]:
if chosen:
    chosen_pixels, chosen_frame, chosen_area = band_predictions(chosen, list(VAL_SETS))
    results, results_frame = evaluate_predictions(chosen_pixels, cases), evaluate_predictions(chosen_frame, cases)
else:
    chosen_area, results, results_frame = {}, baseline, baseline
results_table(results).round(4)

In [ ]:
def summary_table(runs: dict[str, dict]) -> pd.DataFrame:
    """One row per (variant, val set): scalars plus per-class IoU / Boundary IoU."""
    return pd.DataFrame({(name, s): scalar_row(r[s]) for name, r in runs.items() for s in [*VAL_SETS, MEAN_KEY]}).T

In [ ]:
summary = summary_table({"baseline": baseline, "full_frame": full, "band_pixels": results,
                         "band_frame_masked": results_frame})
summary[["miou", "biou", "bf1_1px", "bf1_2px", "iou_track", "biou_track", "iou_building", "biou_building"]].round(4)

# Latency

In [ ]:
tile = standard_tile(val_sets[LATENCY_SET][0].rgb, *LATENCY_TOP_LEFT_PX)
tile_output = lraspp_outputs(model, tile, "cpu")
latency_kwargs = {"warmup": LATENCY_WARMUP, "runs": LATENCY_RUNS, "threads": LATENCY_THREADS}
argmax_latency = refiner_latency(bilinear_argmax, tile_output, tile, **latency_kwargs)
full_latency = refiner_latency(full_fn, tile_output, tile, **latency_kwargs)
argmax_latency.median_ms, full_latency.median_ms

In [ ]:
def latency_row(config: BandConfig) -> dict:
    """Median ms of one band config in pixel mode and in tile mode, with the band's and tiles' area share."""
    tiles = replace(config, crop="tiles")
    area, tile_area = band_fractions(tile_output, tiles)
    return {"band": band_name(config), "area": area, "tile_area": tile_area,
            "pixels_ms": refiner_latency(band_refiner(masked_fn, config), tile_output, tile, **latency_kwargs).median_ms,
            "tiles_ms": refiner_latency(band_refiner(full_fn, tiles), tile_output, tile, **latency_kwargs).median_ms}

In [ ]:
latency = pd.DataFrame([latency_row(c) for c in configs])
if len(latency):
    latency["pixels_to_full"] = latency["pixels_ms"] / full_latency.median_ms
    latency["tiles_to_full"] = latency["tiles_ms"] / full_latency.median_ms
latency.round(3)

# Metrics

In [ ]:
def run_metrics() -> dict[str, float]:
    """Chosen band per set and mean, deltas vs baseline and vs full frame, area, latency, pre-registered checks."""
    variants = {"": results, "full_": full, "frame_": results_frame}
    metrics = {f"{prefix}{k}_{s}".replace("-", "_"): v for prefix, r in variants.items()
               for s in [*VAL_SETS, MEAN_KEY] for k, v in scalar_row(r[s]).items()}
    metrics["boundary_iou_mean"] = results[MEAN_KEY]["biou"]
    for k in ("biou", "miou", "bf1_1px"):
        metrics[f"{k}_gain"] = results[MEAN_KEY][k] - baseline[MEAN_KEY][k]
        metrics[f"full_{k}_gain"] = full[MEAN_KEY][k] - baseline[MEAN_KEY][k]
        metrics[f"band_vs_full_{k}"] = results[MEAN_KEY][k] - full[MEAN_KEY][k]
    metrics.update({f"band_area_{s}".replace("-", "_"): a for s, a in chosen_area.items()})
    metrics["band_area_mean"] = float(np.mean(list(chosen_area.values()))) if chosen_area else 0.0
    metrics.update({"argmax_ms_512": argmax_latency.median_ms, "full_ms_512": full_latency.median_ms,
                    "full_to_argmax": full_latency.median_ms / argmax_latency.median_ms})
    if chosen:
        row = latency[latency["band"] == band_name(chosen)].iloc[0]
        metrics.update({"band_ms_512": row["pixels_ms"], "band_tiles_ms_512": row["tiles_ms"],
                        "band_to_argmax": row["pixels_ms"] / argmax_latency.median_ms,
                        "band_to_full_ms": row["pixels_to_full"], "band_area_latency_tile": row["area"],
                        "tile_area_latency_tile": row["tile_area"],
                        "latency_proportional": bool(row["pixels_to_full"] <= row["area"] + LATENCY_AREA_SLACK)})
    metrics["clears_bar"] = bool(metrics["biou_gain"] >= BIOU_BAR and metrics["miou_gain"] >= -MIOU_GUARD)
    return metrics

In [ ]:
metrics = run_metrics()
{k: round(float(v), 4) for k, v in metrics.items() if k.endswith("_mean") or "ms" in k or "gain" in k
 or k.startswith(("band_", "clears", "latency"))}

# Plots

In [ ]:
def colour(ids: np.ndarray) -> np.ndarray:
    """Class ids to RGB with SEG_PALETTE; ignored pixels white."""
    out = np.full((*ids.shape, 3), 255, np.uint8)
    ok = ids != IGNORE_LABEL
    out[ok] = SEG_PALETTE[ids[ok]]
    return out

In [ ]:
def with_label_edges(pred: np.ndarray, label: np.ndarray) -> np.ndarray:
    """Predicted classes, with the label's 1-px class contours drawn in white."""
    out = colour(pred)
    out[np.any([label_band(label, c, 1) for c in range(len(CLASSES))], axis=0)] = 255
    return out

In [ ]:
def zoom_panels(set_name: str, index: int, box: list[int]) -> list[tuple[np.ndarray, str]]:
    """RGB with the band tinted, label, baseline, full-frame refiner, band refiner, for one crop."""
    case = cases[set_name][index]
    zoom = np.s_[box[0]:box[2], box[1]:box[3]]
    label = case.image.label[zoom]
    rgb = case.image.rgb[zoom].copy()
    if chosen:
        band = uncertain_band(case.output, chosen)[zoom]
        rgb[band] = (0.5 * rgb[band] + 0.5 * np.array([255, 0, 255])).astype(np.uint8)
    band_pred = chosen_pixels[set_name][index] if chosen else bilinear_argmax(case.output, case.image.rgb)
    return [(rgb, f"{set_name} RGB + band"), (colour(label), "label"),
            (with_label_edges(bilinear_argmax(case.output, case.image.rgb)[zoom], label), "baseline"),
            (with_label_edges(full_preds[set_name][index][zoom], label), f"{REFINER} full frame"),
            (with_label_edges(band_pred[zoom], label), f"{REFINER} in band {band_name(chosen) if chosen else '-'}")]

In [ ]:
rows = [zoom_panels(*z) for z in ZOOMS]
fig_zoom, axes = plt.subplots(len(rows), len(rows[0]), figsize=(4 * len(rows[0]), 4 * len(rows)), squeeze=False)
for ax_row, panels in zip(axes, rows):
    for ax, (img, title) in zip(ax_row, panels):
        ax.imshow(img, interpolation="nearest"); ax.set_title(title, fontsize=9); ax.axis("off")
fig_zoom.tight_layout()

In [ ]:
fig_grid, ax = plt.subplots(figsize=(7, 5))
ax.axvline(tune_baseline["miou"] - MIOU_GUARD, color="r", ls="--", lw=1, label="mIoU guard")
ax.scatter(tune_baseline["miou"], tune_baseline["biou"], color="k", marker="s", label="baseline argmax")
ax.scatter(tune_full["miou"], tune_full["biou"], color="tab:red", marker="*", s=120, label=f"{REFINER} full frame")
if len(grid):
    ax.scatter(grid["miou"], grid["biou"], color="tab:blue", label="band, pixel mode")
    ax.scatter(grid["frame_miou"], grid["frame_biou"], color="tab:blue", marker="x", label="band, masked full frame")
    for _, r in grid.iterrows():
        ax.annotate(r["band"], (r["miou"], r["biou"]), fontsize=8, xytext=(3, 3), textcoords="offset points")
ax.set_xlabel(f"mIoU ({TUNE_SET})"); ax.set_ylabel(f"Boundary IoU ({TUNE_SET})"); ax.grid(alpha=.3)
ax.legend(fontsize=8); ax.set_title(REFINER)
fig_grid.tight_layout()

In [ ]:
fig_latency, ax = plt.subplots(figsize=(6, 4.5))
ax.plot([0, 1], [0, 1], color="k", ls=":", lw=1, label="cost = band area")
if len(latency):
    ax.scatter(latency["area"], latency["pixels_to_full"], label="pixel mode")
    ax.scatter(latency["area"], latency["tiles_to_full"], marker="x", label=f"tile mode ({TILE_PX} px)")
    for _, r in latency.iterrows():
        ax.annotate(r["band"], (r["area"], r["pixels_to_full"]), fontsize=8, xytext=(3, 3), textcoords="offset points")
ax.set_xlabel("band area fraction (latency tile)"); ax.set_ylabel("band ms / full-frame ms")
ax.set_title(f"{REFINER}: full frame {full_latency.median_ms:.0f} ms, argmax {argmax_latency.median_ms:.1f} ms")
ax.grid(alpha=.3); ax.legend(fontsize=8)
fig_latency.tight_layout()

# Log

In [ ]:
RUN_CONFIG = {
    "refiner": REFINER,
    "base": {"guided": GUIDED, "crf_best": CRF_BEST, "crf_compat10": CRF_COMPAT10, "watershed": WATERSHED}.get(REFINER),
    "band_distances_px": BAND_DISTANCES_PX, "band_margins": BAND_MARGINS, "tile_px": TILE_PX, "context_px": CONTEXT_PX,
    "chosen_band": asdict(chosen) if chosen else None, "tune_set": TUNE_SET, "miou_guard": MIOU_GUARD,
    "dataset": dataset_ref(DATASET), "model": "model-lraspp-bn-satellite.pt",
    "latency": {"set": LATENCY_SET, "top_left_px": LATENCY_TOP_LEFT_PX, "threads": LATENCY_THREADS, "runs": LATENCY_RUNS},
}

In [ ]:
run = log_run(exp=EXP, config=RUN_CONFIG, metrics=metrics, note=NOTE,
              figures=[fig_zoom, fig_grid, fig_latency], seed=SEED, device=DEVICE) if LOG_RUN else {"run": "dry"}
grid.to_csv(ARTIFACTS / f"grid-{run['run']}.csv", index=False)
latency.to_csv(ARTIFACTS / f"latency-{run['run']}.csv", index=False)
summary.to_csv(ARTIFACTS / f"summary-{run['run']}.csv")
run["run"]